In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import nibabel as nib
import numpy as np


In [ ]:

volume_mask_path = '/content/drive/MyDrive/Pipeline/first_run/Volumes/Person1/upper_right_central_incisor_fdi11.nii.gz'

volume_mask_img = nib.load(volume_mask_path)
volume_mask_data = volume_mask_img.get_fdata()


3.  **Load CT Scan:** Next, we'll load the Computed Tomography (CT) scan. This will also likely be in NIfTI format.

In [ ]:
# --- IMPORTANT: Replace with the actual path to your CT scan file ---
ct_scan_path = '/content/drive/MyDrive/Pipeline/Data/Person1.nii.gz'

ct_scan_img = nib.load(ct_scan_path)
ct_scan_data = ct_scan_img.get_fdata()

In [ ]:
print("--- Metadata của Mask ---")
print(volume_mask_img.header)

print("\n--- Metadata của CT Scan ---")
print(ct_scan_img.header)


In [ ]:
import numpy as np
from scipy import ndimage

# Tạo mask boolean ban đầu (chỉ lấy các vùng có giá trị > 0)
mask_boolean_initial = volume_mask_data > 0

# ĐẶC BIỆT: Lấp đầy các lỗ hổng bên trong không gian 3D của mask
# Điều này đảm bảo lấy được toàn bộ thể tích đặc của răng (giữ lại phần lõi/tủy răng)
mask_boolean = ndimage.binary_fill_holes(mask_boolean_initial)

# Áp dụng mask lên CT scan: giữ nguyên giá trị CT ở vùng có răng, gán -1000 (không khí) cho nền
# Ép kiểu về int16 để giữ đúng định dạng thang xám 16-bit
extracted_dental_volume = np.where(mask_boolean, ct_scan_data, -1000).astype(np.int16)

print(f"Kích thước khối răng đã trích xuất: {extracted_dental_volume.shape}")
print(f"Kiểu dữ liệu: {extracted_dental_volume.dtype}")
print(f"Tổng số voxel của mask gốc: {np.sum(mask_boolean_initial)}")
print(f"Tổng số voxel sau khi giữ lại lõi: {np.sum(mask_boolean)}")

In [ ]:
# Lấy mảng 1 chiều chứa tất cả các giá trị thang xám của riêng phần răng (nơi mask = True)
tooth_hu_values = extracted_dental_volume[mask_boolean]

print("--- Thống kê giá trị thang xám (HU) của khối răng đã tách ---")
print(f"Giá trị HU nhỏ nhất: {tooth_hu_values.min()}")
print(f"Giá trị HU lớn nhất: {tooth_hu_values.max()}")
print(f"Giá trị HU trung bình: {tooth_hu_values.mean():.2f}")

# Có thể in thử 10 giá trị ngẫu nhiên để xem cụ thể
print(f"10 giá trị voxel ngẫu nhiên thuộc khối răng: {np.random.choice(tooth_hu_values, 10)}")

### 4. Cắt xén (Cropping) và Chuẩn hóa vị trí (PCA)
Trước tiên, cắt phần không gian thừa để giảm kích thước ma trận. Sau đó áp dụng PCA để xoay khối răng.

In [ ]:
# 4.1 Cắt xén (Cropping) khối răng dựa trên bounding box của mask

# Tìm tọa độ các voxel thuộc răng
coords = np.argwhere(mask_boolean)

# Xác định giới hạn (min, max) theo 3 trục z, y, x
z_min, y_min, x_min = coords.min(axis=0)
z_max, y_max, x_max = coords.max(axis=0)

# Thêm một chút không gian lề (margin) để khi xoay răng không bị cắt lẹm
margin = 20
z_min = max(0, z_min - margin)
y_min = max(0, y_min - margin)
x_min = max(0, x_min - margin)
z_max = min(extracted_dental_volume.shape[0], z_max + margin + 1)
y_max = min(extracted_dental_volume.shape[1], y_max + margin + 1)
x_max = min(extracted_dental_volume.shape[2], x_max + margin + 1)

# Cắt (Crop) khối răng và mask tương ứng
cropped_volume = extracted_dental_volume[z_min:z_max, y_min:y_max, x_min:x_max]
cropped_mask = mask_boolean[z_min:z_max, y_min:y_max, x_min:x_max]

print(f"Kích thước khối răng sau khi crop: {cropped_volume.shape}")

In [ ]:
from sklearn.decomposition import PCA
from scipy.ndimage import affine_transform

# 4.2 Lấy tọa độ không gian 3D của răng để tính PCA
tooth_coords = np.argwhere(cropped_mask)

# Áp dụng PCA để tìm 3 trục chính (principal components)
pca = PCA(n_components=3)
pca.fit(tooth_coords)

# pca.components_ chứa 3 vector chỉ phương của 3 trục chính.
# Trục có phương sai lớn nhất (chiều dài nhất của răng) sẽ nằm ở hàng đầu tiên (index 0).
rotation_matrix = pca.components_

# Để xoay ảnh bằng scipy, ta cần ma trận biến đổi ngược (inverse matrix)
# Do ma trận xoay là ma trận trực giao, nghịch đảo của nó chính là ma trận chuyển vị (transpose)
inv_rotation_matrix = np.linalg.inv(rotation_matrix)

# Tính toán offset để xoay quanh tâm của khối đã crop
center = np.array(cropped_volume.shape) / 2.0
offset = center - np.dot(inv_rotation_matrix, center)

print("Đang tiến hành xoay khối răng... (có thể mất vài giây)")
# Xoay toàn bộ khối răng đã crop. Những vùng trống do xoay sinh ra sẽ được gán bằng -1000 (cval)
rotated_volume = affine_transform(
    cropped_volume,
    inv_rotation_matrix,
    offset=offset,
    order=1, # Dùng nội suy tuyến tính (bilinear interpolation)
    cval=-1000
).astype(np.int16)

print(f"Kích thước khối răng sau khi xoay PCA: {rotated_volume.shape}")

### 5. Trực quan hóa lát cắt 2D tương tác
Sử dụng `ipywidgets` và `matplotlib` để tạo thanh trượt, giúp bạn cuộn và xem chi tiết từng lát cắt (slice) của khối răng theo các trục không gian 3D.

In [ ]:
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# Hàm để vẽ lát cắt dựa trên trục và chỉ số lát cắt
def plot_slice(axis, slice_idx):
    plt.figure(figsize=(6, 6))

    if axis == 'Trục Z (Đứng)':
        # Cắt theo trục Z (chiều cao của răng)
        slice_data = rotated_volume[slice_idx, :, :]
        plt.xlabel('Trục X')
        plt.ylabel('Trục Y')
    elif axis == 'Trục Y (Ngang 1)':
        # Cắt theo trục Y
        slice_data = rotated_volume[:, slice_idx, :]
        plt.xlabel('Trục X')
        plt.ylabel('Trục Z')
    else: # Trục X (Ngang 2)
        # Cắt theo trục X
        slice_data = rotated_volume[:, :, slice_idx]
        plt.xlabel('Trục Y')
        plt.ylabel('Trục Z')

    # Dùng cmap='gray' để xem ảnh xám (như X-quang/CT)
    # vmin, vmax chỉnh độ tương phản (có thể thay đổi tùy ý)
    plt.imshow(slice_data, cmap='gray', vmin=-1000, vmax=2000)
    plt.title(f'Lát cắt dọc theo {axis} - Vị trí: {slice_idx}')
    plt.axis('off')
    plt.show()

# Tạo widget chọn trục
axis_dropdown = widgets.Dropdown(
    options=['Trục Z (Đứng)', 'Trục Y (Ngang 1)', 'Trục X (Ngang 2)'],
    value='Trục Z (Đứng)',
    description='Chọn Trục:'
)

# Tạo thanh trượt
slice_slider = widgets.IntSlider(
    min=0,
    max=rotated_volume.shape[0] - 1,
    step=1,
    value=rotated_volume.shape[0] // 2,
    description='Lát cắt:'
)

# Hàm cập nhật giới hạn thanh trượt khi đổi trục
def update_slider_range(*args):
    if axis_dropdown.value == 'Trục Z (Đứng)':
        slice_slider.max = rotated_volume.shape[0] - 1
    elif axis_dropdown.value == 'Trục Y (Ngang 1)':
        slice_slider.max = rotated_volume.shape[1] - 1
    else:
        slice_slider.max = rotated_volume.shape[2] - 1
    slice_slider.value = slice_slider.max // 2  # Đặt lại giá trị về giữa

axis_dropdown.observe(update_slider_range, 'value')

# Kết nối các widget với hàm vẽ
widgets.interact(plot_slice, axis=axis_dropdown, slice_idx=slice_slider);

### 6. Trực quan hóa Heatmap Tương tác với Plotly (Thang Xám)
Sử dụng `plotly.graph_objects` kết hợp `ipywidgets` để tạo một bảng điều khiển tương tác (Dashboard) mượt mà hơn. Di chuột lên ảnh để xem chính xác giá trị HU của từng voxel.

In [ ]:
from google.colab import output
output.enable_custom_widget_manager()

In [ ]:
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display
import numpy as np

# Lấy kích thước tối đa của khối lượng đã xoay
z_max, y_max, x_max = rotated_volume.shape

# Khởi tạo FigureWidget để cập nhật dữ liệu mượt mà
fig = go.FigureWidget()

# Khởi tạo Heatmap với lát cắt giữa của trục Z
initial_slice = rotated_volume[z_max // 2, :, :]
heatmap = go.Heatmap(
    z=initial_slice,
    colorscale='gray',
    zmin=0,    # Ngưỡng thấp mặc định
    zmax=1500, # Ngưỡng cao mặc định
    hoverinfo='x+y+z',
    colorbar=dict(title='HU')
)
fig.add_trace(heatmap)

# Cấu hình giao diện đồ thị
fig.update_layout(
    title='Trình xem Lát cắt CT (Heatmap Thang xám) - Lọc Ngưỡng HU',
    width=700,
    height=700,
    xaxis=dict(title='Trục X', scaleanchor="y", scaleratio=1),
    yaxis=dict(title='Trục Y', autorange='reversed', scaleanchor="x", scaleratio=1)
)

# --- Tạo Widgets ---
axis_dropdown_pl = widgets.Dropdown(
    options=['Trục Z (Đứng)', 'Trục Y (Ngang 1)', 'Trục X (Ngang 2)'],
    value='Trục Z (Đứng)',
    description='Chọn Trục:'
)

slice_slider_pl = widgets.IntSlider(
    min=0,
    max=z_max - 1,
    step=1,
    value=z_max // 2,
    description='Lát cắt:',
    layout=widgets.Layout(width='500px')
)

# Thanh trượt kép để chỉnh Windowing
hu_range_slider = widgets.IntRangeSlider(
    value=[0, 1500],
    min=-1000,
    max=3500,
    step=50,
    description='Windowing:',
    layout=widgets.Layout(width='500px')
)

# Các widget cho tính năng phân định Threshold
enable_threshold = widgets.Checkbox(
    value=False,
    description='Bật Phân ngưỡng',
    layout=widgets.Layout(width='150px')
)

threshold_slider = widgets.IntSlider(
    min=-1000,
    max=3000,
    step=50,
    value=500,
    description='Ngưỡng lọc:',
    layout=widgets.Layout(width='340px')
)

# --- Hàm cập nhật ---
def update_plotly_view(*args):
    axis = axis_dropdown_pl.value
    idx = slice_slider_pl.value
    hu_min, hu_max = hu_range_slider.value
    is_thresh = enable_threshold.value
    thresh_val = threshold_slider.value

    if axis == 'Trục Z (Đứng)':
        slice_data = rotated_volume[idx, :, :]
        fig.layout.xaxis.title = 'Trục X'
        fig.layout.yaxis.title = 'Trục Y'
    elif axis == 'Trục Y (Ngang 1)':
        slice_data = rotated_volume[:, idx, :]
        fig.layout.xaxis.title = 'Trục X'
        fig.layout.yaxis.title = 'Trục Z'
    else: # Trục X (Ngang 2)
        slice_data = rotated_volume[:, :, idx]
        fig.layout.xaxis.title = 'Trục Y'
        fig.layout.yaxis.title = 'Trục Z'

    # Nếu bật phân ngưỡng, các giá trị < thresh_val sẽ bị chìm vào nền (hu_min - 1000)
    if is_thresh:
        display_data = np.where(slice_data >= thresh_val, slice_data, hu_min - 1000)
    else:
        display_data = slice_data

    # Cập nhật dữ liệu Z và ngưỡng HU
    with fig.batch_update():
        fig.data[0].z = display_data
        fig.data[0].zmin = hu_min
        fig.data[0].zmax = hu_max

def update_plotly_slider_range(*args):
    axis = axis_dropdown_pl.value
    if axis == 'Trục Z (Đứng)':
        slice_slider_pl.max = z_max - 1
    elif axis == 'Trục Y (Ngang 1)':
        slice_slider_pl.max = y_max - 1
    else:
        slice_slider_pl.max = x_max - 1

    slice_slider_pl.value = slice_slider_pl.max // 2
    update_plotly_view()

# Liên kết các event
axis_dropdown_pl.observe(update_plotly_slider_range, 'value')
slice_slider_pl.observe(update_plotly_view, 'value')
hu_range_slider.observe(update_plotly_view, 'value')
enable_threshold.observe(update_plotly_view, 'value')
threshold_slider.observe(update_plotly_view, 'value')

# Hiển thị giao diện
ui = widgets.VBox([
    widgets.HBox([axis_dropdown_pl, slice_slider_pl]),
    hu_range_slider,
    widgets.HBox([enable_threshold, threshold_slider]),
    fig
])
display(ui)
